# Scenario cookbook lab

This notebook implements three provider-neutral recipes for the internal HR/IT support assistant: typed expense extraction, agent writes, and moderation. Every fixture is synthetic and deterministic; moderation scores are frozen fixture data, not a live classifier. The support-assistant recipe remains covered by Course 01.

In [ ]:
import sys
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from cookbook_lab import (
    BudgetExceeded,
    Decision,
    Executor,
    ExpenseClaimV2,
    GuardOutcome,
    ProposedAction,
    approve_gate,
    dry_run,
    evaluate_by_category,
    identity_hash,
    load_json,
    moderate,
    route_extraction,
    run_write_recipe,
)

FIXTURES = Path("fixtures")
print("deterministic cookbook simulation:", FIXTURES)


## 1. Recipe map

Each recipe places controls at the boundary where they can be enforced and records evidence for review. The support recipe's tenant filtering, provenance, citations, and output handling are already executable in Course 01, so this lab concentrates on extraction, external writes, and moderation. The same typed outcomes make the recipes comparable without requiring a model provider.

In [ ]:
recipe_map = {
    "support": "Course 01: tenant-aware retrieval and grounded output",
    "extraction": "schema + invariants + field confidence + review routing",
    "agent_write": "dry run + approval + idempotency + verification + reconcile",
    "moderation": "precheck + category thresholds + FP/FN measurement",
}
recipe_map


## 2. Data extraction pipeline

`ExpenseClaimV2` is versioned and rejects unknown fields before any persistence decision. Invariants are deterministic: line items must add to the total, currency must be allowed, and dates cannot be in the future; the minimum confidence across extracted fields routes uncertain claims to review. Source spans remain attached to each typed field so a reviewer can inspect provenance.

In [ ]:
extraction_policy = load_json(FIXTURES / "extraction_policy.json")
extraction_candidates = load_json(FIXTURES / "extraction_candidates.json")
extraction_results = {
    item["id"]: route_extraction(item["candidate"], extraction_policy, date(2026, 12, 31))
    for item in extraction_candidates
}
{key: (value.decision.value, value.reason_codes) for key, value in extraction_results.items()}


## 3. Agent that writes to external systems

The write recipe previews the proposed tool and arguments before checking the allowlist, requester role, approval, and kill switch. `Executor` applies a write budget and idempotency key, then verifies the recorded state; a mismatch becomes `reconcile_required` rather than being reported as success. Reads and low-risk ticket creation can be allowed while payroll changes require an authorized HR approver.

In [ ]:
agent_policy = load_json(FIXTURES / "agent_policy.json")
agent_items = load_json(FIXTURES / "agent_actions.json")
executor = Executor(agent_policy["budgets"]["max_writes_per_session"])
agent_results = {}
for item in agent_items:
    action = ProposedAction(**item["action"])
    preview = dry_run(action, agent_policy)
    expected_state = {"tool": action.tool, **action.arguments}
    if item.get("expected_state_mismatch"):
        expected_state["amount"] = 999
    outcomes = run_write_recipe(action, agent_policy, executor, expected_state)
    agent_results[item["id"]] = {
        "preview": preview,
        "outcomes": [(outcome.decision.value, outcome.reason_codes) for outcome in outcomes],
    }
agent_results


## 4. Content moderation gateway

The gateway runs deterministic size, MIME, tenant, and rate checks before using frozen category scores. Thresholds are versioned per category, and the highest severity wins: block, escalate, transform as a warning, then allow. The category report treats each label as a positive and counts only outcomes driven by that category, making false positives and false negatives visible.

In [ ]:
moderation_policy = load_json(FIXTURES / "moderation_policy.json")
moderation_posts = load_json(FIXTURES / "moderation_posts.json")
moderation_counts = evaluate_by_category(moderation_posts, moderation_policy)
moderation_counts


## 5. Scenario-specific release checklist

The checklist below is filled from evidence produced by the agent-write recipe rather than from prose claims. Preview, approval, idempotency, verification, and reconciliation are demonstrated by the recipe outcomes; the remaining items identify the operational evidence that must accompany a release. A checklist item is useful only when its evidence path is explicit.

In [ ]:
release_checklist = [
    ("owner and harm model", True, "agent_policy.version and scenario owner"),
    ("trust boundaries", True, "approve_gate enforces requester and approver roles"),
    ("authorization outside model", True, "agent_policy.allowlist"),
    ("preview and approval", True, "dry_run and approve_gate"),
    ("idempotency and reconciliation", True, "Executor receipts and verify"),
    ("budget and kill switch", True, "agent_policy.budgets and kill_switch"),
    ("synthetic regression tests", True, "agent_actions.json and test_scenario_cookbook.py"),
    ("minimized logs", True, "receipt contains action id and key, not raw prompts"),
]
release_checklist


## 6. Exercises

1. Add an extraction candidate with no line items and predict the invariant route.
2. Turn on the agent policy kill switch and explain which writes stop while reads remain available.
3. Add a moderation category or threshold and update its per-category FP/FN regression test.

All exercises should remain offline, deterministic, and tied to a fixture or explicit evidence path.